In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Style global ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f6',
    'axes.grid': True,
    'grid.color': '#e8e8e5',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'xtick.bottom': False,
    'ytick.left': False,
    'axes.labelcolor': '#555',
    'xtick.color': '#555',
    'ytick.color': '#555',
})

# ── Palette ───────────────────────────────────────────────
BLUE   = '#378ADD'
GRAY   = '#B4B2A9'
GREEN  = '#3B6D11'
RED    = '#A32D2D'
ORANGE = '#BA7517'

RES_COLOR = {'W': GREEN, 'D': ORANGE, 'L': RED}

OUTPUT_DIR = '../outputs/projet2_charts'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Librairies chargées')

In [ ]:
df_periods = pd.read_csv('../data/03_matches_phases.csv')

# Sépare FBBP et adversaires
df_per_fbbp = df_periods[df_periods['team'] == 'FBBP'].copy().reset_index(drop=True)
df_per_adv  = df_periods[df_periods['team'] != 'FBBP'].copy().reset_index(drop=True)

# Ordre des périodes
period_order = ['1-15', '16-30', '31-45+', '46-60', '61-75', '76-90+']
df_per_fbbp['period'] = pd.Categorical(df_per_fbbp['period'], categories=period_order, ordered=True)
df_per_adv['period']  = pd.Categorical(df_per_adv['period'],  categories=period_order, ordered=True)

# Résultats par match
results = {
    'FBBP_ORL_20260220': ('L', 'Orléans'),
    'FBBP_ROU_20260306': ('W', 'Rouen'),
    'FBBP_SBR_20260321': ('D', 'Briochin'),
    'QRM_FBBP_20260228': ('D', 'QRM'),
    'VER_FBBP_20260313': ('L', 'Versailles'),
}
df_per_fbbp['resultat'] = df_per_fbbp['match_id'].map(lambda x: results[x][0])
df_per_fbbp['adversaire'] = df_per_fbbp['match_id'].map(lambda x: results[x][1])
df_per_adv['resultat']  = df_per_adv['match_id'].map(lambda x: results[x][0])
df_per_adv['adversaire'] = df_per_adv['match_id'].map(lambda x: results[x][1])

print(df_per_fbbp[['match_id','period','ppda','attacks_per_min','recoveries_per_min','possession_pct']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14))

match_colors = {
    'FBBP_ORL_20260220': (RED,    'Orléans (L)'),
    'FBBP_ROU_20260306': (GREEN,  'Rouen (W)'),
    'FBBP_SBR_20260321': (ORANGE, 'Briochin (D)'),
    'QRM_FBBP_20260228': (ORANGE, 'QRM (D)'),
    'VER_FBBP_20260313': (RED,    'Versailles (L)'),
}

x = np.arange(len(period_order))

# ── Graphique 1 : PPDA par période ──────────────────────
ax1 = axes[0]
for match_id, (color, label) in match_colors.items():
    data = df_per_fbbp[df_per_fbbp['match_id'] == match_id].sort_values('period')
    ls = '--' if color == RED else '-'
    ax1.plot(x, data['ppda'].values, color=color, linewidth=2,
             linestyle=ls, marker='o', markersize=5, label=label)

ax1.axhline(10, color='#888', linewidth=1, linestyle=':', alpha=0.7)
ax1.text(5.5, 10.3, 'seuil ~10', fontsize=9, color='#888', ha='right')
ax1.set_xticks(x)
ax1.set_xticklabels(period_order, fontsize=10)
ax1.set_ylabel('PPDA', fontsize=11)
ax1.set_title('Dynamique du pressing — PPDA par période', fontsize=12, pad=8, fontweight='bold')
ax1.legend(fontsize=9, frameon=False, loc='upper left')

# ── Graphique 2 : Attaques par minute ───────────────────
ax2 = axes[1]
for match_id, (color, label) in match_colors.items():
    data = df_per_fbbp[df_per_fbbp['match_id'] == match_id].sort_values('period')
    ls = '--' if color == RED else '-'
    ax2.plot(x, data['attacks_per_min'].values, color=color, linewidth=2,
             linestyle=ls, marker='o', markersize=5, label=label)

ax2.set_xticks(x)
ax2.set_xticklabels(period_order, fontsize=10)
ax2.set_ylabel('Attaques / min', fontsize=11)
ax2.set_title('Intensité offensive FBBP — Attaques par minute', fontsize=12, pad=8, fontweight='bold')
ax2.legend(fontsize=9, frameon=False, loc='upper left')

# ── Graphique 3 : Moyenne PPDA tous matchs ──────────────
ax3 = axes[2]
avg_ppda_fbbp = df_per_fbbp.groupby('period')['ppda'].mean().reindex(period_order)
avg_ppda_adv  = df_per_adv.groupby('period')['ppda'].mean().reindex(period_order)

ax3.plot(x, avg_ppda_fbbp.values, color=BLUE,   linewidth=2.5,
         marker='o', markersize=7, label='PPDA FBBP (moy.)')
ax3.plot(x, avg_ppda_adv.values,  color=GRAY,   linewidth=2.5,
         marker='s', markersize=7, label='PPDA adversaire (moy.)')

for i, (v1, v2) in enumerate(zip(avg_ppda_fbbp.values, avg_ppda_adv.values)):
    ax3.text(i, v1 + 0.5, f'{v1:.1f}', ha='center', fontsize=9,
             color=BLUE, fontweight='bold')
    ax3.text(i, v2 - 1.5, f'{v2:.1f}', ha='center', fontsize=9,
             color='#888', fontweight='bold')

ax3.axhline(10, color='#888', linewidth=1, linestyle=':', alpha=0.7)
ax3.set_xticks(x)
ax3.set_xticklabels(period_order, fontsize=10)
ax3.set_ylabel('PPDA moyen', fontsize=11)
ax3.set_title('PPDA moyen FBBP vs adversaires — tous matchs', fontsize=12, pad=8, fontweight='bold')
ax3.legend(fontsize=9, frameon=False)

fig.suptitle('FBBP — Analyse des phases de jeu (J22 → J26)',
             fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phases_jeu.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Graphique 1 : Danger début de match ─────────────────
ax1 = axes[0]

# PPDA FBBP vs adversaire sur 1-15 et 16-30
periods_debut = ['1-15', '16-30']
x = np.arange(len(periods_debut))
w = 0.35

ppda_fbbp_debut = df_per_fbbp[df_per_fbbp['period'].isin(periods_debut)].groupby('period')['ppda'].mean().reindex(periods_debut)
ppda_adv_debut  = df_per_adv[df_per_adv['period'].isin(periods_debut)].groupby('period')['ppda'].mean().reindex(periods_debut)
atk_adv_debut   = df_per_adv[df_per_adv['period'].isin(periods_debut)].groupby('period')['attacks_per_min'].mean().reindex(periods_debut)

b1 = ax1.bar(x - w/2, ppda_fbbp_debut.values, w, label='PPDA FBBP',
             color=BLUE, edgecolor='white', linewidth=1)
b2 = ax1.bar(x + w/2, ppda_adv_debut.values,  w, label='PPDA adversaire',
             color=GRAY, edgecolor='white', linewidth=1)

for bar, val in zip(b1, ppda_fbbp_debut.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', fontsize=10, color=BLUE, fontweight='bold')
for bar, val in zip(b2, ppda_adv_debut.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}', ha='center', fontsize=10, color='#888')

ax1.axhline(10, color='#888', linewidth=1, linestyle=':', alpha=0.7)
ax1.set_xticks(x)
ax1.set_xticklabels(periods_debut, fontsize=11)
ax1.set_ylabel('PPDA moyen', fontsize=11)
ax1.set_title('Zone de danger — 30 premières minutes\nFBBP presse peu, adversaires attaquent fort',
              fontsize=11, pad=8, fontweight='bold')
ax1.legend(fontsize=9, frameon=False)

# ── Graphique 2 : Attaques par minute toutes périodes ───
ax2 = axes[1]

avg_atk_fbbp = df_per_fbbp.groupby('period')['attacks_per_min'].mean().reindex(period_order)
avg_atk_adv  = df_per_adv.groupby('period')['attacks_per_min'].mean().reindex(period_order)

xp = np.arange(len(period_order))
ax2.plot(xp, avg_atk_fbbp.values, color=BLUE, linewidth=2.5,
         marker='o', markersize=7, label='Attaques FBBP (moy.)')
ax2.plot(xp, avg_atk_adv.values,  color=GRAY, linewidth=2.5,
         marker='s', markersize=7, label='Attaques adversaires (moy.)')

for i, (v1, v2) in enumerate(zip(avg_atk_fbbp.values, avg_atk_adv.values)):
    ax2.text(i, v1 + 0.015, f'{v1:.2f}', ha='center', fontsize=9,
             color=BLUE, fontweight='bold')
    ax2.text(i, v2 - 0.03, f'{v2:.2f}', ha='center', fontsize=9,
             color='#888')

# Zone danger début
ax2.axvspan(-0.5, 1.5, alpha=0.06, color=RED, zorder=0)
ax2.text(0.5, ax2.get_ylim()[1] if ax2.get_ylim()[1] else 0.9,
         'zone\ndanger', ha='center', fontsize=8, color=RED, alpha=0.7)

ax2.set_xticks(xp)
ax2.set_xticklabels(period_order, fontsize=10)
ax2.set_ylabel('Attaques / minute', fontsize=11)
ax2.set_title('Intensité offensive — FBBP vs adversaires\npar tranche de 15 min',
              fontsize=11, pad=8, fontweight='bold')
ax2.legend(fontsize=9, frameon=False)

fig.suptitle('FBBP — Patterns de phases de jeu',
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phases_patterns.png', dpi=150, bbox_inches='tight')
plt.show()